# Process events with shared state

## Goal

Some decisions depend on what happened earlier. Usage limits, running balances, and account-level eligibility need a current value that changes as events arrive. This is **stateful processing**.

Here, a flow reads an account balance, applies one event, and writes the new balance back. The next event for that account sees the updated value. Rulebricks holds the state and decision logic; Spark supplies ordered events and saves their outcomes.

| Account A event | Balance before | Change | Balance after |
| --- | --- | --- | --- |
| 1 | 0 | +100 | 100 |
| 2 | 100 | -25 | 75 |
| 3 | 75 | +50 | 125 |

Events for the same account run in order. Independent accounts can run in parallel. The saved result is an event ledger plus a final state observation.

## Setup

Follow the [installation guide](../docs/getting-started.md). Edit the API URL, flow identifier, published version, context name, and output location in the cell below. The next two steps describe the flow and its state store. Use shared output storage on a cluster; the local path is for local Spark.

Start this state-changing job with speculation disabled, `spark.task.maxFailures=1`, and `spark.stage.maxConsecutiveAttempts=1`. The SDK checks these settings before flow execution; configure them when the cluster or session starts. Give this job exclusive access to its account keys. The API key needs context creation, write, read, and flow execution permissions.

In [ ]:
import os
import uuid

from pyspark.sql import SparkSession, functions
from rulebricks.types import ContextSchema, ContextSchemaField

from rulebricks_spark import contexts

spark = (
    SparkSession.getActiveSession()
    or SparkSession.builder.appName("stateful-processing")
    .config("spark.speculation", "false")
    .config("spark.task.maxFailures", "1")
    .config("spark.stage.maxConsecutiveAttempts", "1")
    .getOrCreate()
)
assert spark.sparkContext.getConf().get("spark.speculation", "false").lower() == "false"
assert spark.sparkContext.getConf().get("spark.task.maxFailures", "4") == "1"
assert spark.sparkContext.getConf().get("spark.stage.maxConsecutiveAttempts", "4") == "1"
api_key = os.environ["RULEBRICKS_API_KEY"]
base_url = "https://rulebricks.acme.com/api/v1"
flow_slug = "account-balance"
flow_version = "1"
context_name = "Spark account balances"
output_root = "file:///tmp/rulebricks-examples"
run_id = str(uuid.uuid4())
run_path = f"{output_root}/{run_id}"
ledger_path = f"{run_path}/events"
state_path = f"{run_path}/state"
print("Run output:", run_path)
seed_path = f"{run_path}/initial-state"

## 1. Prepare account state

A Rulebricks **context** is a named store of facts. One context definition can hold many account records, called **instances**. The key `account_id` identifies which account owns each balance.

| Stored field | Purpose |
| --- | --- |
| `account_id` | Identity of the state owner |
| `balance` | Current value, initially zero |

`ensure()` creates the definition if needed or checks an existing one. Automatic triggering stays off because Spark will explicitly call the flow.

In [ ]:
context_slug = contexts.ensure(
    context_name,
    schema=ContextSchema(
        base=[
            ContextSchemaField(key="account_id", name="Account", type="string"),
            ContextSchemaField(key="balance", name="Balance", type="number", default_value=0),
        ]
    ),
    key="account_id",
    ttl_seconds=86400,
    history_limit=1,
    api_key=api_key,
    base_url=base_url,
)
print("Use this context in the flow's operation steps:", context_slug)

## 2. Connect the flow to the state

Configure the published flow with the printed context identifier:

| Flow step | What it does |
| --- | --- |
| Accept input | Receive `account_id`, `event_id`, and `amount` |
| Read Context | Read that account's current balance |
| Calculate | Add the event's amount to the balance |
| Update Context | Store the new balance for the same account |
| Return result | Return `account_id`, `event_id`, and `balance` |

The Context Operation steps perform the reads and writes. No context binding is needed. A returned balance alone would not update stored state.

## 3. Supply events and their order

The example includes two accounts and deliberately scrambles their events. `sequence` gives a unique order within each account.

| Source field | Role |
| --- | --- |
| `account_id` | Groups events sharing one state record |
| `sequence` | Determines their order; retained in the ledger |
| `event_id` | Matches each outcome to an event |
| `amount` | The change applied by the flow |

Use stable business IDs and unique, non-null ordering values in your own pipeline.

In [ ]:
events = spark.createDataFrame(
    [
        (f"{run_id}-A", 2, "A-2", -25),
        (f"{run_id}-B", 2, "B-2", 20),
        (f"{run_id}-A", 1, "A-1", 100),
        (f"{run_id}-A", 3, "A-3", 50),
        (f"{run_id}-B", 1, "B-1", 40),
    ],
    "account_id string, sequence long, event_id string, amount long",
)
expected_count = events.count()

## 4. Initialize missing accounts

Submit just the distinct account IDs. Rulebricks creates missing instances with a zero balance and preserves existing balances.

| Account exists? | Result of identity-only submission |
| --- | --- |
| No | Create the account using the default balance |
| Yes | Retain its current balance |

Sending `balance=0` explicitly would reset an existing balance. Check the saved receipts before processing events.

In [ ]:
initialized = contexts.ingest(
    events.select("account_id").distinct(),
    context_slug,
    key="account_id",
    api_key=api_key,
    base_url=base_url,
)
initialized.write.format("delta").mode("errorifexists").save(seed_path)
seeds = spark.read.format("delta").load(seed_path)
assert seeds.count() == 2
if seeds.filter("rulebricks_meta.status != 'ok'").limit(1).count():
    raise RuntimeError("Reconcile initial state receipts before processing events.")

## 5. Process the events and save the ledger

`contexts.apply_flow()` groups events by account, sorts them, and waits for each flow response before sending the next event for that account.

| Ledger columns | Contents |
| --- | --- |
| Source fields | The event and its ordering information |
| `rulebricks_result` | Flow result as JSON |
| `rulebricks_meta` | Outcome, execution IDs, timing, and errors |

A failed event blocks later events for that account. An uncertain request stops the remaining partition. There are no automatic HTTP retries. If your flow uses custom failure messages, supply a `ResponseOptions.error_handler` so they are recognized before later events run.

In [ ]:
evaluated = contexts.apply_flow(
    events,
    flow_slug,
    version=flow_version,
    correlation_id_path="event_id",
    key="account_id",
    order_by=["sequence"],
    input_columns=["account_id", "event_id", "amount"],
    execution_partitions=2,
    api_key=api_key,
    base_url=base_url,
)
evaluated.write.format("delta").mode("errorifexists").save(ledger_path)
ledger = spark.read.format("delta").load(ledger_path)
assert ledger.count() == expected_count
ledger.select(
    "account_id",
    "sequence",
    "event_id",
    "rulebricks_meta.status",
    "rulebricks_meta.error",
    "rulebricks_meta.trace_id",
).orderBy("account_id", "sequence").show(truncate=False)

## 6. Inspect the saved event outcomes

Select the returned balance as a column for inspection. This reads the saved ledger and makes no flow calls. Account A should progress through 100, 75, and 125; account B through 40 and 60.

The execution ID helps locate the corresponding run in Rulebricks when an event needs investigation.

In [ ]:
decisions = ledger.select(
    "account_id",
    "sequence",
    "event_id",
    functions.get_json_object("rulebricks_result", "$.data.balance").alias("balance"),
    "rulebricks_meta.execution_id",
    "rulebricks_meta.trace_id",
    "rulebricks_result",
)
decisions.orderBy("account_id", "sequence").show(truncate=False)
if ledger.filter("rulebricks_meta.status != 'ok'").limit(1).count():
    raise RuntimeError(
        "Reconcile failed or blocked events from the saved ledger before continuing."
    )

## 7. Verify the stored state

A flow response describes its execution. A state lookup checks what is stored now. Save that observation and compare it with the expected final balances.

| Account | Expected balance |
| --- | --- |
| A | 125 |
| B | 60 |

In [ ]:
observed = contexts.lookup(
    ledger.select("account_id").distinct(),
    context_slug,
    key="account_id",
    execution_partitions=2,
    api_key=api_key,
    base_url=base_url,
)
observed.write.format("delta").mode("errorifexists").save(state_path)
saved_state = spark.read.format("delta").load(state_path)
assert saved_state.filter("rulebricks_meta.status != 'ok'").count() == 0
balances = saved_state.select(
    "account_id",
    functions.get_json_object("rulebricks_result", "$.state.balance").cast("long").alias("balance"),
)
expected = spark.createDataFrame(
    [(f"{run_id}-A", 125), (f"{run_id}-B", 60)], ["account_id", "balance"]
)
assert balances.exceptAll(expected).count() == 0
assert expected.exceptAll(balances).count() == 0
balances.orderBy("account_id").show(truncate=False)

## Next steps

Choose the key according to the state being shared. One key per account allows accounts to run independently. One constant key represents a global state record and requires sequential execution.

Ordering inside one job does not coordinate other writers. A repeated action or restarted job can repeat writes, and Spark output commit is separate from Rulebricks writes. For production, design flow effects to tolerate replay or reconcile interrupted runs before resubmitting events. For automatic recovery from a caller-owned checkpoint, use the [fault tolerance notebook](06-fault-tolerance.ipynb). `resume(saved_ledger)` cannot safely replay isolated failed stateful events.

This additive example teaches ordering; it is not an exactly-once ledger. Keep durable event history in your data lake or warehouse, and choose context retention to cover the processing window. See [context ordering and recovery](../docs/contexts.md#preserve-ordering-when-something-goes-wrong).